In [1]:
library(progress)
install.packages("dbarts")
install.packages("stochtree")
install.packages("mvtnorm")
library(mvtnorm)
library(stochtree)
library(dbarts)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘BH’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dbarts’


The following object is masked from ‘package:stochtree’:

    bart




# DGP_2

In [2]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
results_matrix <- matrix(NA, nrow = num_simulations, ncol = 30)
colnames(results_matrix) <- c("bcf_1k_pehe1", "bcf_1k_pehe2","bcf_0.5k_pehe1", "bcf_0.5k_pehe2","bcf_0.25k_pehe1", "bcf_0.25k_pehe2","bcf_0.1k_pehe1", "bcf_0.1k_pehe2",
                             "bcf_0.05k_pehe1", "bcf_0.05k_pehe2",
                             "bcf_1k_tau_951", "bcf_1k_tau_952","bcf_0.5k_tau_951", "bcf_0.5k_tau_952","bcf_0.25k_tau_951", "bcf_0.25k_tau_952","bcf_0.1k_tau_951", "bcf_0.1k_tau_952",
                             "bcf_0.05k_tau_951", "bcf_0.05k_tau_952", "bcf_1k_tau_951w", "bcf_1k_tau_952w","bcf_0.5k_tau_951w", "bcf_0.5k_tau_952w","bcf_0.25k_tau_951w", "bcf_0.25k_tau_952w","bcf_0.1k_tau_951w", "bcf_0.1k_tau_952w",
                             "bcf_0.05k_tau_951w", "bcf_0.05k_tau_952w")

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-runif(n)
X2<-runif(n)
X3<-runif(n)
X4<-runif(n)
X5<-runif(n)
X6<-rbinom(n, 1, 0.5)
X7<-rbinom(n, 1, 0.5)
X8<-rbinom(n, 1, 0.5)
X9<-sample(c(0, 1, 2, 3, 4), n, replace=T)
X10<-sample(c(0, 1, 2, 3, 4), n, replace=T)

X<-cbind(X1, X2, X3, X5, X6, X7, X8, X9, X10)

Mu1<-(11*sin(pi*X1*X2)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
Mu2<-(9*sin(pi*X1*X2)+22*(X3-0.5)^2+0*X4+8*X6+X9)*10+300

Tau1<-(2*X4+2*X5)*10
Tau2<-(1*X4+3*X5)*10

true_propensity<-X4

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-runif(n_test)
X2_test<-runif(n_test)
X3_test<-runif(n_test)
X4_test<-runif(n_test)
X5_test<-runif(n_test)
X6_test<-rbinom(n_test, 1, 0.5)
X7_test<-rbinom(n_test, 1, 0.5)
X8_test<-rbinom(n_test, 1, 0.5)
X9_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)
X10_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)

X_test<-cbind(X1_test, X2_test, X3_test, X5_test, X6_test, X7_test, X8_test, X9_test, X10_test)

Mu1_test<-(11*sin(pi*X1_test*X2_test)+18*(X3_test-0.5)^2+10*X4_test+12*X6_test+X9_test)*10+300
Mu2_test<-(9*sin(pi*X1_test*X2_test)+22*(X3_test-0.5)^2+0*X4_test+8*X6_test+X9_test)*10+300

Tau1_test<-(2*X4_test+2*X5_test)*10
Tau2_test<-(1*X4_test+3*X5_test)*10

true_propensity_test<-X4_test

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-1000
n_burn<-500

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

bcf_1k_mod1<-bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_1k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_1k_mod1$tau_hat_test))^2))

bcf_1k_mod2<-bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_1k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_1k_mod2$tau_hat_test))^2))

bcf_1k_tau_951<-mean(diag(apply(bcf_1k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))
bcf_1k_tau_952<-mean(diag(apply(bcf_1k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_1k_tau_951w<-mean(apply(bcf_1k_mod1$tau_hat_test, 1, cred_width, 0.95))
bcf_1k_tau_952w<-mean(apply(bcf_1k_mod2$tau_hat_test, 1, cred_width, 0.95))

n_iter<-500
n_burn<-250

bcf_0.5k_mod1<-bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.5k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.5k_mod1$tau_hat_test))^2))

bcf_0.5k_mod2<-bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.5k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.5k_mod2$tau_hat_test))^2))

bcf_0.5k_tau_951<-mean(diag(apply(bcf_0.5k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))
bcf_0.5k_tau_952<-mean(diag(apply(bcf_0.5k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.5k_tau_951w<-mean(apply(bcf_0.5k_mod1$tau_hat_test, 1, cred_width, 0.95))
bcf_0.5k_tau_952w<-mean(apply(bcf_0.5k_mod2$tau_hat_test, 1, cred_width, 0.95))

n_iter<-250
n_burn<-125

bcf_0.25k_mod1<-bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.25k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.25k_mod1$tau_hat_test))^2))

bcf_0.25k_mod2<-bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.25k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.25k_mod2$tau_hat_test))^2))

bcf_0.25k_tau_951<-mean(diag(apply(bcf_0.25k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))
bcf_0.25k_tau_952<-mean(diag(apply(bcf_0.25k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.25k_tau_951w<-mean(apply(bcf_0.25k_mod1$tau_hat_test, 1, cred_width, 0.95))
bcf_0.25k_tau_952w<-mean(apply(bcf_0.25k_mod2$tau_hat_test, 1, cred_width, 0.95))

n_iter<-100
n_burn<-50

bcf_0.1k_mod1<-bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.1k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.1k_mod1$tau_hat_test))^2))

bcf_0.1k_mod2<-bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.1k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.1k_mod2$tau_hat_test))^2))

bcf_0.1k_tau_951<-mean(diag(apply(bcf_0.1k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))
bcf_0.1k_tau_952<-mean(diag(apply(bcf_0.1k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.1k_tau_951w<-mean(apply(bcf_0.1k_mod1$tau_hat_test, 1, cred_width, 0.95))
bcf_0.1k_tau_952w<-mean(apply(bcf_0.1k_mod2$tau_hat_test, 1, cred_width, 0.95))

n_iter<-50
n_burn<-25

bcf_0.05k_mod1<-bcf(X2, Z, Y[,1], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.05k_pehe1<-sqrt(mean((Tau1_test-rowMeans(bcf_0.05k_mod1$tau_hat_test))^2))

bcf_0.05k_mod2<-bcf(X2, Z, Y[,2], X_test = X2_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = 200)
bcf_0.05k_pehe2<-sqrt(mean((Tau2_test-rowMeans(bcf_0.05k_mod2$tau_hat_test))^2))

bcf_0.05k_tau_951<-mean(diag(apply(bcf_0.05k_mod1$tau_hat_test, 1, in_cred, Tau1_test, 0.95)))
bcf_0.05k_tau_952<-mean(diag(apply(bcf_0.05k_mod2$tau_hat_test, 1, in_cred, Tau2_test, 0.95)))

bcf_0.05k_tau_951w<-mean(apply(bcf_0.05k_mod1$tau_hat_test, 1, cred_width, 0.95))
bcf_0.05k_tau_952w<-mean(apply(bcf_0.05k_mod2$tau_hat_test, 1, cred_width, 0.95))

# Store the results in the matrix
  results_matrix[i, ] <- c(bcf_1k_pehe1, bcf_1k_pehe2, bcf_0.5k_pehe1, bcf_0.5k_pehe2, bcf_0.25k_pehe1, bcf_0.25k_pehe2, bcf_0.1k_pehe1, bcf_0.1k_pehe2,
                             bcf_0.05k_pehe1, bcf_0.05k_pehe2,
                             bcf_1k_tau_951, bcf_1k_tau_952, bcf_0.5k_tau_951, bcf_0.5k_tau_952, bcf_0.25k_tau_951, bcf_0.25k_tau_952, bcf_0.1k_tau_951, bcf_0.1k_tau_952,
                             bcf_0.05k_tau_951, bcf_0.05k_tau_952, bcf_1k_tau_951w, bcf_1k_tau_952w, bcf_0.5k_tau_951w, bcf_0.5k_tau_952w, bcf_0.25k_tau_951w, bcf_0.25k_tau_952w, bcf_0.1k_tau_951w, bcf_0.1k_tau_952w,
                             bcf_0.05k_tau_951w, bcf_0.05k_tau_952w)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "wsBCF_simulation_results_DGP2.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to wsBCF_simulation_results_DGP2.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position”
Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position”
Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position”
Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position”
Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
“column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, 

Simulation completed and results saved to wsBCF_simulation_results_DGP2.csv


In [3]:
print(results_matrix)

       bcf_1k_pehe1 bcf_1k_pehe2 bcf_0.5k_pehe1 bcf_0.5k_pehe2 bcf_0.25k_pehe1
  [1,]     41.87690     9.179580       40.91444       9.202680        42.25533
  [2,]     46.40463    12.580820       47.60860      13.642107        46.27142
  [3,]     37.28976     8.997822       35.33259       6.737789        36.89047
  [4,]     50.46432     9.512393       44.12423       8.922559        46.15470
  [5,]     40.57397     8.191614       37.71574       8.502871        36.10289
  [6,]     48.17464    10.318028       48.59880      11.817428        65.56062
  [7,]     42.73463     9.733884       39.29873       8.801312        39.45720
  [8,]     35.67131     8.479292       36.18211      10.351955        41.65180
  [9,]     40.62020     8.052111       40.37548       6.616144        44.30205
 [10,]     35.30785    11.064232       33.88517      10.872242        35.67402
 [11,]     36.75662    11.174127       35.73731       9.973336        34.69217
 [12,]     37.58125    11.307854       35.21256     